# Inverted Index

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Hash Tables, Strings · **Difficulty/Frequency:** Popular! (10/10)

## Concepts

**What this problem is really testing:**
- Hash maps (`dict`) and hash sets (`set`) — the whole solution is built out of them
- The *inverted* index idea: flipping "document → its words" into "word → the documents containing it"
- Set algebra (intersection / union) as the implementation of boolean AND / OR queries

**First-principles primer — what is each piece?**

- **Hash map (`dict`)** — a container of `key -> value` pairs where finding a key takes constant time on average, because the key is run through a hash function that computes *where* the value lives instead of searching for it. Here: `token -> the set of documents containing that token`.
- **Hash set (`set`)** — the same machinery with no values attached; it answers "is this in here?" in O(1) and supports fast `&` (intersection) and `|` (union). Here: each token's **posting list** is a set of document IDs.
- **Posting list** — search-engine jargon for "the list of documents a term appears in". It is the value side of the inverted index.
- **Tokenization** — cutting a document string into the searchable units (here: lowercase, whitespace-split words). Normalizing at *both* index time and query time is what makes `"Coffee"` and `"coffee"` match.

**Why the "inverted" framing matters:**

A normal (forward) index answers *"what words are in document 7?"*. That is the wrong direction for search — a query gives you a **word** and wants **documents**. Answering it with a forward index means scanning every document on every query: O(N × L). Building the reverse mapping once, at insert time, turns every later search into a single dictionary lookup.

**The AND-ordering trick:** for `advancedSearch([...], "AND")`, intersect the **smallest** posting list first. Intersection can only shrink the result, so starting small means every later intersection scans a tiny set. If `"the"` appears in 10,000 docs and `"mongodb"` in 3, starting from the 3 saves ~10,000 comparisons.

**Simple worked example.** Insert `"Coffee is good"` (id 0) and `"I am at a coffee shop"` (id 1):

| token | posting list |
|---|---|
| `coffee` | `{0, 1}` |
| `is` | `{0}` |
| `good` | `{0}` |
| `i`, `am`, `at`, `a`, `shop` | `{1}` |

`search("coffee")` → look up one key → `{0, 1}` → `["Coffee is good", "I am at a coffee shop"]`. No document was ever scanned.

## Problem Statement

Implement an `InvertedIndex` supporting:

| Method | Behaviour |
|---|---|
| `insert(doc)` | Index the document string; assign it an internal id |
| `search(token)` | Return every currently-indexed document containing `token` |
| `delete(doc)` | Remove that document (by its exact original string) from the index |
| `advancedSearch(tokens, op)` | `op="AND"` → docs containing **all** tokens; `op="OR"` → docs containing **any** token |

**Example**

```python
idx = InvertedIndex()
idx.insert("Coffee is good")
idx.insert("I am at a coffee shop")

idx.search("coffee")                              # -> both documents
idx.delete("Coffee is good")
idx.search("coffee")                              # -> ["I am at a coffee shop"]

idx.advancedSearch(["coffee", "shop"], "AND")     # -> ["I am at a coffee shop"]
idx.advancedSearch(["coffee", "shop"], "OR")      # -> ["I am at a coffee shop"] (after the delete)
```

Matching is **case-insensitive** (`"Coffee"` matches `"coffee"`), and results are returned in insertion order so the output is deterministic.

### Approach 1 — Naive (keep the documents, scan them on every search)

**Idea:** store the raw document strings in a list. On `search`, walk every document, tokenize it, and check whether the query token is present. There is no precomputation at all, so `insert` is trivially cheap and `search` pays the full price every single time.

**Time complexity:** O(1) per `insert`; **O(N × L) per `search`** (N documents, L tokens each) — every query re-reads the entire corpus. `delete` is O(N).

**Space complexity:** O(N × L) — just the documents themselves.

In [ ]:
from typing import Dict, List, Set
from collections import defaultdict


class NaiveIndex:
    """Baseline: no index at all - every search re-scans every document."""

    def __init__(self) -> None:
        self.docs: List[str] = []

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return text.lower().split()               # normalize once, here and at query time

    def insert(self, doc: str) -> None:
        self.docs.append(doc)                     # O(1) - all the cost is deferred to search

    def delete(self, doc: str) -> None:
        if doc in self.docs:
            self.docs.remove(doc)                 # O(N) scan to find it

    def search(self, query: str) -> List[str]:
        token = query.lower().strip()
        # O(N * L): tokenize and scan EVERY document, on EVERY call
        return [d for d in self.docs if token in self._tokenize(d)]

    def advancedSearch(self, tokens: List[str], operator: str) -> List[str]:
        wanted = {t.lower() for t in tokens}
        if not wanted:
            return []
        out = []
        for d in self.docs:
            present = set(self._tokenize(d))
            if operator == "AND" and wanted <= present:
                out.append(d)
            elif operator == "OR" and (wanted & present):
                out.append(d)
            elif operator not in ("AND", "OR"):
                raise ValueError(f"Unsupported operator: {operator}")
        return out

### Approach 2 — Optimal (the inverted index: `token -> set of doc ids`)

**Idea:** pay the tokenization cost **once**, at insert time, and store the reverse mapping. Two dictionaries carry the whole design:

- `index: token -> set[doc_id]` — the posting lists
- `docs: doc_id -> original string` — so we can hand back the text the caller inserted

Storing **integer ids** (not strings) in the posting lists is what makes `advancedSearch` cheap: small integers hash fast and set intersection/union work directly on them.

`delete` finds the id by matching the original string (O(N) worst case — a reverse `string -> id` map would make it O(1), worth mentioning aloud), then removes that id from each of the document's tokens and drops any posting list that became empty. Leaving empty sets behind is a real bug: `search` would then return `[]` from a key that "exists", and the index would leak memory over time.

**Time complexity:** O(L) per `insert`; **O(1) average per `search`** plus O(k log k) to sort the k results; O(N + L) per `delete`.

**Space complexity:** O(N × L_avg) — one id per token occurrence.

In [ ]:
class InvertedIndex:
    """token -> set of doc ids, plus doc id -> original text."""

    def __init__(self) -> None:
        self.index: Dict[str, Set[int]] = defaultdict(set)   # token -> posting list
        self.docs: Dict[int, str] = {}                       # doc id -> original string
        self.next_id = 0

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return text.lower().split()

    def insert(self, doc: str) -> int:
        doc_id = self.next_id
        self.next_id += 1                        # ids are monotonic => sorting by id == insertion order
        self.docs[doc_id] = doc
        for token in self._tokenize(doc):
            self.index[token].add(doc_id)        # O(1) per token
        return doc_id

    def delete(self, doc: str) -> None:
        doc_id = None
        for did, text in self.docs.items():      # O(N) - a reverse map would make this O(1)
            if text == doc:
                doc_id = did
                break
        if doc_id is None:
            return                               # deleting something absent is a no-op
        for token in self._tokenize(doc):
            if token in self.index:
                self.index[token].discard(doc_id)
                if not self.index[token]:
                    del self.index[token]        # never leave empty posting lists behind
        del self.docs[doc_id]

    def search(self, query: str) -> List[str]:
        token = query.lower().strip()
        if token not in self.index:
            return []
        return [self.docs[d] for d in sorted(self.index[token])]   # sorted id == insertion order

    def advancedSearch(self, tokens: List[str], operator: str) -> List[str]:
        toks = [t.lower() for t in tokens]
        if not toks:
            return []
        if operator == "AND":
            # Smallest posting list first: intersection only shrinks, so start small.
            toks.sort(key=lambda t: len(self.index.get(t, ())))
            result = set(self.index.get(toks[0], set()))
            for t in toks[1:]:
                result &= self.index.get(t, set())
                if not result:
                    break                        # early exit - nothing can come back
        elif operator == "OR":
            result = set()
            for t in toks:
                result |= self.index.get(t, set())
        else:
            raise ValueError(f"Unsupported operator: {operator}")
        return [self.docs[d] for d in sorted(result)]

## Verification

Run the exact scenario from the problem statement, then cross-check the naive baseline and the indexed implementation agree on randomised data and on the documented edge cases.

In [ ]:
# --- The scenario straight from the problem statement ---
idx = InvertedIndex()
idx.insert("Coffee is good")
idx.insert("I am at a coffee shop")

assert idx.search("coffee") == ["Coffee is good", "I am at a coffee shop"]
assert idx.search("Coffee") == ["Coffee is good", "I am at a coffee shop"]   # case-insensitive

idx.delete("Coffee is good")
assert idx.search("coffee") == ["I am at a coffee shop"]

# Re-insert so both AND and OR have something to work with
idx.insert("Coffee is good")
assert idx.advancedSearch(["coffee", "shop"], "AND") == ["I am at a coffee shop"]
assert sorted(idx.advancedSearch(["coffee", "shop"], "OR")) == sorted(
    ["I am at a coffee shop", "Coffee is good"]
)

# --- Edge cases ---
assert idx.search("nonexistent") == []                 # unknown token
assert idx.advancedSearch([], "AND") == []             # empty token list
assert idx.advancedSearch(["coffee", "zzz"], "AND") == []   # one token matches nothing

empty = InvertedIndex()
assert empty.search("anything") == []                  # empty index
empty.delete("not there")                              # deleting an absent doc is a no-op

try:
    idx.advancedSearch(["coffee"], "XOR")
except ValueError:
    pass
else:
    raise AssertionError("unsupported operator should raise ValueError")

# Deleting must not leave an empty posting list behind
tiny = InvertedIndex()
tiny.insert("unique token here")
tiny.delete("unique token here")
assert "unique" not in tiny.index, "empty posting lists must be cleaned up"
assert tiny.docs == {}

# --- Both implementations agree on randomised data ---
import random

random.seed(7)
vocab = ["coffee", "shop", "mongo", "index", "query", "shard", "replica"]
corpus = [" ".join(random.choices(vocab, k=random.randint(2, 6))) for _ in range(200)]
corpus = list(dict.fromkeys(corpus))               # de-duplicate: delete-by-string needs unique docs

naive, fast = NaiveIndex(), InvertedIndex()
for d in corpus:
    naive.insert(d)
    fast.insert(d)

for token in vocab + ["missing"]:
    assert sorted(naive.search(token)) == sorted(fast.search(token))

for op in ("AND", "OR"):
    for _ in range(50):
        q = random.sample(vocab, k=random.randint(1, 3))
        assert sorted(naive.advancedSearch(q, op)) == sorted(fast.advancedSearch(q, op)), (q, op)

# ...and they still agree after deletions
for d in corpus[:50]:
    naive.delete(d)
    fast.delete(d)
for token in vocab:
    assert sorted(naive.search(token)) == sorted(fast.search(token))

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Duplicate document text.** `delete(doc)` matches on the exact string, so two identical documents make "which one?" ambiguous — the loop removes the first id it finds. Cleaner contracts: return the id from `insert` and delete by id, or keep `text -> list[id]` and define delete as "remove one occurrence" (or "remove all"). Say which you picked and why.
- **Phrase search (`"coffee shop"` as an exact phrase).** Posting lists lose word order. Store **positions** instead: `token -> {doc_id: [positions]}`. A phrase matches when, for consecutive query tokens, some position `p` of the first is followed by `p+1` of the second in the same document. Space grows to one integer per token occurrence, which is why real engines make positional indexes optional.
- **Relevance ranking (TF-IDF / BM25).** Membership isn't enough — you need **term frequency** per document (how often the token appears there) and **document frequency** (`len(posting_list)`) for the inverse-document-frequency weight. Store counts instead of a bare set: `token -> {doc_id: count}`, then score and sort instead of returning insertion order.
- **Thread safety.** A single global lock serialises everything. Finer-grained options: shard the index by `hash(token) % k` with one lock per shard (writers to different tokens never contend), or use a copy-on-write posting list so readers never block. The subtle part is that `advancedSearch` touches several tokens and therefore several locks — take them in a fixed order (e.g. sorted by token) to avoid deadlock.
- **Prefix search (`"coff*"`).** Add a **trie** over the vocabulary whose nodes point at posting lists. Walking `c-o-f-f` lands on a subtree; union the posting lists beneath it. Alternative: keep the vocabulary in a sorted list and use `bisect` to find the prefix range — simpler, and fine when the vocabulary fits in memory.

## Empirical complexity check

Compare the **naive per-query scan** (Approach 1, O(N × L) every call) against the **inverted index** (Approach 2, O(N × L) once at build time, then O(1) per call). Both run the same fixed number of queries (`Q = 50`) against a corpus that doubles in size.

| Growth when n doubles | What it means |
|---|---|
| ~2x | linear — total work scales with corpus size (naive: each of the 50 queries costs O(N)) |
| ~2x, but far smaller constant | the index's one-time build; the 50 lookups themselves cost essentially nothing |

The ratios for both end up near 2x — the honest result, since building the index is itself linear. What the **absolute times** show is the point: the index pays O(N) *once* and then answers unlimited queries for free, while the naive version pays O(N) *per query*. Raise `Q` and the gap widens without bound; that is the real lesson.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random

QUERIES = 50
VOCAB = [f"w{i}" for i in range(60)]


def make_corpus(n):
    rng = random.Random(1)
    return ([" ".join(rng.choices(VOCAB, k=8)) for _ in range(n)],)


def run_naive(corpus):
    ix = NaiveIndex()
    for d in corpus:
        ix.insert(d)
    for q in range(QUERIES):
        ix.search(VOCAB[q % len(VOCAB)])          # O(N * L) EVERY call


def run_indexed(corpus):
    ix = InvertedIndex()
    for d in corpus:
        ix.insert(d)                              # O(L) once per document...
    for q in range(QUERIES):
        ix.search(VOCAB[q % len(VOCAB)])          # ...then O(1) per query


benchmark(
    {"Approach 1 - naive scan per query": run_naive,
     "Approach 2 - inverted index": run_indexed},
    make_corpus,
    sizes=[500, 1000, 2000, 4000],
    repeats=3,
)

## Patterns learned

- **Invert the mapping to match the query direction.** Whenever lookups go "value → which containers hold it?", build that map at write time instead of scanning at read time. Same trick behind database secondary indexes, reverse adjacency lists in graphs, and `Counter`-style tallies.
- **Store ids, not payloads, in the index.** Integer ids are cheap to hash, cheap to intersect, and keep exactly one copy of the real data in a side table. This is the same normalisation idea as a foreign key.
- **Boolean queries are set algebra.** AND is `&`, OR is `|`, NOT is `-`. Once you see it that way, the implementation writes itself — and so do the optimisations.
- **Order your intersections smallest-first.** Any operation that can only shrink a result should start from the smallest input. Cheap to add, large constant-factor win, and interviewers watch for it.
- **Deleting means restoring the invariant everywhere.** State the invariant ("`index[t]` holds exactly the live docs containing `t`") and then check every method preserves it. Empty posting lists left behind are the classic bug this catches.
- **Sort by a monotonic id to get deterministic output for free.** Because ids are assigned in increasing order, `sorted(ids)` *is* insertion order — no extra bookkeeping, no reliance on set iteration order.